In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error
import statsmodels.api as sm
import pickle
import os

# -----------------------------
# 1. Load dataset
# -----------------------------
file_path = "/home/andrew/Desktop/python_scripts/house-price-prediction-api/backend/src/house/House Price Prediction Dataset.csv"
if not os.path.exists(file_path):
    raise FileNotFoundError(f"File not found: {file_path}")

df = pd.read_csv(file_path)

# -----------------------------
# 2. Rename columns if needed
# -----------------------------
df.rename(columns={"YearBuilt": "Year_Built"}, inplace=True)

# -----------------------------
# 3. Select relevant features
# -----------------------------
selected_features = ["Area", "Bedrooms", "Bathrooms", "Floors", "Year_Built", "Location", "Condition", "Garage", "Price"]
df = df[selected_features]

# -----------------------------
# 4. Handle missing values
# -----------------------------
print("Missing values per column:\n", df.isnull().sum())

# Fill numeric columns with mean
numeric_cols = ["Area", "Bedrooms", "Bathrooms", "Floors", "Year_Built"]
df[numeric_cols] = df[numeric_cols].fillna(df[numeric_cols].mean())

# Optional: Drop rows with missing categorical data
categorical_cols = ["Location", "Condition", "Garage"]
df.dropna(subset=categorical_cols, inplace=True)

# -----------------------------
# 5. Encode categorical variables
# -----------------------------
# One-Hot Encoding for Location and Condition
df = pd.get_dummies(df, columns=["Location", "Condition"], drop_first=True)

# Convert Garage to numeric
df["Garage"] = df["Garage"].map({"Yes": 1, "No": 0})

# -----------------------------
# 6. Separate features and target
# -----------------------------
X = df.drop("Price", axis=1).astype(float)
y = df["Price"].astype(float)

# -----------------------------
# 7. Optional: OLS regression for p-values
# -----------------------------
X_sm = sm.add_constant(X)  # Adds intercept term
model_sm = sm.OLS(y, X_sm).fit()
print(model_sm.summary())

# -----------------------------
# 8. Split dataset and train model
# -----------------------------
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

model = LinearRegression()
model.fit(X_train, y_train)

# -----------------------------
# 9. Evaluate model
# -----------------------------
y_pred = model.predict(X_test)
print("R²:", round(r2_score(y_test, y_pred), 4))
print("MAE:", round(mean_absolute_error(y_test, y_pred), 2))
print("MSE:", round(mean_squared_error(y_test, y_pred), 2))
print("RMSE:", round(np.sqrt(mean_squared_error(y_test, y_pred)), 2))

# -----------------------------
# 10. Save model
# -----------------------------
output_model_path = "model.pkl"
with open(output_model_path, "wb") as f:
    pickle.dump(model, f)

print(f"Model saved successfully as '{output_model_path}'")